<div style="background-color: #F4F6F7; padding: 20px; border-radius: 8px; font-family: 'Arial', sans-serif; color: #2E4053;">
  <!-- Logo centrado -->
  <div style="text-align: center; margin-bottom: 15px;">
    <img src="https://drive.google.com/uc?export=view&id=1kjzXfjTiieYAd4azw5bh4UfEg91gUIdh" alt="Logo Institucional" width="250" />
  </div>

  <!-- Título principal -->
  <h1 style="font-size: 36px; font-weight: bold; text-align: right;">
    <strong>Estimación del gradiente geotérmico a partir de imágenes multiespectrales del LANDSAT 7 y base de datos tabular</strong>
  </h1>

  <!-- Datos del estudiante -->
  <p style="font-size: 22px; text-align: center; margin: 5px 0;">
    <strong>Nombre:</strong> Carlos Enrique Montilla Morales &nbsp;|&nbsp;
    <strong>Institución:</strong> Universidad Tecnológica de Pereira  &nbsp;|&nbsp;
    <strong>Grupo de investigación en Automática </strong>
  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />

  <!-- Descripción breve -->
  <p style="font-size: 18px; text-align: center; font-style: italic; margin: 5px 0;">
    Este cuaderno es la primera prueba donde se entrena un modelo multimodal mediante el uso de imágenes satelitales multiespectrales del LANDSAT7 y una base de datos que contienen diversas variables geológicas y geofísicas para estimar el gradiente geotérmico en Colombia. Incluye: ingestión y preprocesado de mosaicos (manejo de NoData), creación de muestras/parches, estrategia de fine-tuning, evaluación con métricas de regresión R2, MAE y RMSE y exportación de predicciones y checkpoints para análisis geoespacial.

  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />
</div>


# **1. Carga de las librerías, imágenes y la base de datos**

In [ ]:
!pip install -q --upgrade pip
!pip install -q rasterio==1.4 pyproj rioxarray
!pip install -q torchvision tqdm scikit-image pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 35.1 MB/s eta 0:00:00


In [ ]:
#!gdown 1jxJkZG92IpRO8GWkLS3C3u_zk0ILLb45 # Base de datos Min-Max
!gdown 1OIJY8QtxRYr4HW504Awxama22OAuBJID # Base de datos cruda
#!gdown 1BAm52wXxOT0DeZBbgv9FgDrVCfMFrUF_ #Geotiffs
#!gdown 1oZAsFy_yDXnJUbUQbuIMlIS-SFaEhd_X #Geotiffs completos
!gdown 1CnCNvGQSoWNEIbNz-j5bIa_COHpK15-l #Geotiffs completos 256

!unzip -q LANDSAT_GEOTIFF_COMPLETE_256.zip -d LANDSAT_GEOTIFF

Downloading...
From: https://drive.google.com/uc?id=1OIJY8QtxRYr4HW504Awxama22OAuBJID
To: /content/data_prep.csv
100% 1.01M/1.01M [00:00<00:00, 60.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1CnCNvGQSoWNEIbNz-j5bIa_COHpK15-l
From (redirected): https://drive.google.com/uc?id=1CnCNvGQSoWNEIbNz-j5bIa_COHpK15-l&confirm=t&uuid=a836d913-939a-4aeb-9efc-a9f25eeb023d
To: /content/LANDSAT_GEOTIFF_COMPLETE_256.zip
100% 4.26G/4.26G [01:00<00:00, 69.9MB/s]


In [ ]:
# Cell 2
import os, re, time, joblib
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import json

import rasterio
from scipy.spatial import cKDTree
from skimage.transform import resize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torchvision.models as models
import geopandas as gpd

BASE_DIR = Path("/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF")  # carpeta con LANDSAT7_*.tif
#CSV_BASE = Path("/content/normalized_data_multimodal2.csv")                         # CSV base con 'ID' y la columna objetivo
CSV_BASE = Path("/content/data_prep.csv")                         # CSV base con 'ID' y la columna objetivo
TARGET_COL = "Apparent Geothermal Gradient (°C/Km)"          # Variable  objetivo
OUT_DIR = Path("/content/run_resnet50_se_maps")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# ---------------------------------

TARGET_SIZE = (256,256)     # (H,W) de entrada para la CNN (reduce si GPU limitada)
FILL_METHOD_IMAGE = 'knn' # 'median' o 'knn'
K_NEIGHBORS = 8
IMAGE_NORM = 'zcore'       # 'minmax' o 'zscore'
TAB_IMPUTE_METHOD = 'knn'   # 'median' o 'knn'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
NUM_WORKERS = 2

EPOCHS_HEAD = 3       # epochs solo cabeza
EPOCHS_BACKBONE = 4  # epochs descongelado (layer4 / backbone)
EPOCHS_FULL = 3       # opcional: descongelar todo
LR_HEAD = 5e-4
LR_FINE = 5e-5
WEIGHT_DECAY = 1e-5
PATIENCE = 2

print("Device:", DEVICE)

Device: cuda


In [ ]:
# Cell 3
def read_geotiff(path: Path):
    with rasterio.open(str(path)) as ds:
        arr = ds.read().astype(float)  # (bands, H, W)
        nodata = ds.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        return arr, ds.meta

def fill_missing_median(arr):
    out = arr.copy()
    for b in range(out.shape[0]):
        band = out[b]
        mask = np.isnan(band)
        if mask.any():
            med = np.nanmedian(band)
            if np.isnan(med):
                med = 0.0
            band[mask] = med
            out[b] = band
    return out

def fill_missing_knn_mean(arr, k=10):
    out = arr.copy()
    bands, H, W = out.shape
    for b in range(bands):
        band = out[b]
        mask_nan = np.isnan(band)
        if not mask_nan.any():
            continue
        ys, xs = np.where(~mask_nan)
        if len(ys) == 0:
            band[mask_nan] = 0.0
            out[b] = band
            continue
        valid_coords = np.column_stack((ys, xs))
        valid_vals = band[ys, xs]
        tree = cKDTree(valid_coords)
        ys_nan, xs_nan = np.where(mask_nan)
        nan_coords = np.column_stack((ys_nan, xs_nan))
        k_use = min(k, len(valid_coords))
        dists, idxs = tree.query(nan_coords, k=k_use, workers=-1)
        if k_use == 1:
            idxs = idxs[:, None]
        vals_neigh = valid_vals[idxs]
        mean_vals = np.nanmean(vals_neigh, axis=1)
        band[ys_nan, xs_nan] = mean_vals
        out[b] = band
    return out

def resize_multiband(arr, target_size=TARGET_SIZE):
    bands, H, W = arr.shape
    Ht, Wt = target_size

    if (H, W) == (Ht, Wt):
        return arr

    else:
        arr_t = np.transpose(arr, (1,2,0))
        resized = resize(
            arr_t, (Ht, Wt, bands),
            order=1,
            preserve_range=True,
            anti_aliasing=True
        )
        return np.transpose(resized, (2,0,1))


def normalize_image_minmax(arr, eps=1e-8):
    out = arr.astype(np.float32).copy()
    for b in range(out.shape[0]):
        band = out[b]
        vmin = np.nanmin(band)
        vmax = np.nanmax(band)
        denom = (vmax - vmin) if (vmax - vmin) > eps else 1.0
        out[b] = (band - vmin) / denom
    return out

def normalize_image_zscore(arr, eps=1e-8):
    out = arr.astype(np.float32).copy()
    for b in range(out.shape[0]):
        band = out[b]
        mean = np.nanmean(band)
        std = np.nanstd(band)
        denom = std if std > eps else 1.0
        out[b] = (band - mean) / denom
    return out


In [ ]:
# Cell 4
df_base = pd.read_csv(CSV_BASE, delimiter=';')
df_base = df_base.drop(columns=['Unnamed: 23'])
if 'ID' not in df_base.columns:
    raise ValueError("CSV_BASE debe contener columna 'ID'.")

df = df_base.copy()
df['ID'] = df['ID'].astype(str).str.strip().str.replace(',', '.', regex=False)
df['ID_numeric'] = pd.to_numeric(df['ID'], errors='coerce')
if df['ID_numeric'].isna().any():
    def extract_first_int(s):
        m = re.search(r'\d+', str(s))
        return int(m.group(0)) if m else np.nan
    df.loc[df['ID_numeric'].isna(), 'ID_numeric'] = df.loc[df['ID_numeric'].isna(), 'ID'].apply(extract_first_int)
if df['ID_numeric'].isna().any():
    raise ValueError("Algunas filas no pudieron convertirse a ID entero.")
df['ID'] = df['ID_numeric'].astype(int)
df.drop(columns=['ID_numeric'], inplace=True)

df['tif_name'] = df['ID'].apply(lambda x: f"LANDSAT7_{int(x)}.tif")
df['file_exists'] = df['tif_name'].apply(lambda fn: (BASE_DIR / fn).exists())
n_missing = (~df['file_exists']).sum()
print(f"Total registros: {len(df)}  TIFF faltantes: {n_missing}")
if n_missing > 0:
    print("Ejemplos faltantes:", df.loc[~df['file_exists'], ['ID','tif_name']].head(10).to_string(index=False))
    df = df.loc[df['file_exists']].reset_index(drop=True)
    print("Se eliminaron filas sin TIFF. Registros restantes:", len(df))

FINAL_CSV = OUT_DIR / "final_dataset.csv"
df_out = df.drop(columns=['tif_name','file_exists'], errors='ignore')
df_out.to_csv(FINAL_CSV, index=False)
print("CSV final guardado en:", FINAL_CSV)

Total registros: 4543  TIFF faltantes: 0
CSV final guardado en: /content/run_resnet50_se_maps/final_dataset.csv


In [ ]:
df_final = pd.read_csv(FINAL_CSV)

extreme_gradient_threshold = df_final['Apparent Geothermal Gradient (°C/Km)'].quantile(0.99)
print(f"Extreme gradient threshold (99th percentile): {extreme_gradient_threshold}")

lowest_gradient_threshold = df_final['Apparent Geothermal Gradient (°C/Km)'].quantile(0.01)
print(f"Lowest gradient threshold (01st percentile): {lowest_gradient_threshold}")

df_final = df_final[df_final['Apparent Geothermal Gradient (°C/Km)'] <= extreme_gradient_threshold]
df_final = df_final[df_final['Apparent Geothermal Gradient (°C/Km)'] >= lowest_gradient_threshold]

df_final.info()

Extreme gradient threshold (99th percentile): 43.808904607399995
Lowest gradient threshold (01st percentile): 14.8731798686
<class 'pandas.core.frame.DataFrame'>
Index: 4451 entries, 1 to 4542
Data columns (total 23 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Latitude                              4451 non-null   float64
 1   Longitude                             4451 non-null   float64
 2   Elevation (m)                         4451 non-null   float64
 3   Surface Temperature (°C)              4451 non-null   float64
 4   Apparent Geothermal Gradient (°C/Km)  4451 non-null   float64
 5   Moho Depth (m)                        4451 non-null   float64
 6   Magnetic Anomaly (nT)                 4451 non-null   float64
 7   Fault                                 4451 non-null   float64
 8   Strike-slip Fault                     4451 non-null   float64
 9   Reverse or Thrust Fault         

In [ ]:
# # Define bin edges and assign each gradient value to a bin
# bins = np.linspace(df_final['Apparent Geothermal Gradient (°C/Km)'].min(),
#                    df_final['Apparent Geothermal Gradient (°C/Km)'].max(),
#                    num=16)  # 50 bins + 1 for the edge
# df_final['Gradient Bin'] = np.digitize(df_final['Apparent Geothermal Gradient (°C/Km)'], bins)

# # Calculate the frequency of each bin
# bin_counts = df_final['Gradient Bin'].value_counts()

# # Calculate the raw weights
# raw_weights = (1 / (bin_counts + 0.00001))

# # Normalize the raw weights so the smallest weight (most frequent bin) is 1
# normalized_weights = raw_weights / raw_weights.min()

# # Now clip the maximum weight to 50
# clipped_weights = np.clip(normalized_weights, 1, 50)

# # Map the clipped weights back to the dataframe
# df_final['Gradient Weight'] = df_final['Gradient Bin'].map(clipped_weights)

# # Normalize the weights again to have the maximum as 1
# df_final['Gradient Weight'] = df_final['Gradient Weight'] / df_final['Gradient Weight'].max()

# # Drop the 'Gradient Bin' column if it's no longer needed
# df_final.drop('Gradient Bin', axis=1, inplace=True)

In [ ]:
# from scipy.stats import gaussian_kde
# import numpy as np

# y = df_final['Apparent Geothermal Gradient (°C/Km)'].values

# kde = gaussian_kde(y)
# density = kde(y)

# weights = 1 / 2 * np.sqrt(density)


# #weights = np.clip(weights, 0, 5)

# weights = weights / weights.mean()

# df_final['Gradient Weight'] = weights

# print("Max weight:", weights.max())
# print("Min weight:", weights.min())

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d

y = df_final['Apparent Geothermal Gradient (°C/Km)'].values

# Discretizar en bins
n_bins = int(np.sqrt(len(y)))
hist, bin_edges = np.histogram(y, bins=n_bins)

# Suavizar distribución (LDS)
sigma = 30
smoothed_hist = gaussian_filter1d(hist.astype(float), sigma=sigma)

# Asignar densidad a cada muestra
bin_ids = np.digitize(y, bin_edges[:-1]) - 1
bin_ids = np.clip(bin_ids, 0, n_bins-1)

density = smoothed_hist[bin_ids]

# pesos
weights = 1 / density


# normalización
weights = weights / weights.mean()

df_final['Gradient Weight'] = weights

print("Max weight:", weights.max())
print("Min weight:", weights.min())

Max weight: 1.8000319391208581
Min weight: 0.8488798454351674


In [ ]:
# from scipy.stats import gaussian_kde
# import numpy as np

# coords = df_final[['Latitude', 'Longitude']].values.T

# kde = gaussian_kde(coords)

# spatial_density = kde(coords)

# spatial_weights = 1 / spatial_density

# spatial_weights = spatial_weights / spatial_weights.mean()

# df_final['Spatial Weight'] = spatial_weights

In [ ]:
# df_final['Final Weight'] = (
#     df_final['Gradient Weight'] *
#     df_final['Spatial Weight']
# )

# df_final['Final Weight'] = df_final['Final Weight'] / df_final['Final Weight'].mean()

In [ ]:
#weights = df_final['Final Weight']
weights = df_final['Gradient Weight'].values
sample_weights = torch.tensor(weights, dtype=torch.float)

df_final = df_final[['Latitude', 'Longitude', 'Elevation (m)',
       'Curie Depth (Km)','Moho Depth (m)',
       'Strike-slip Fault', 'Reverse or Thrust Fault',
       'Right-lateral Fault', 'Nearest Basement',
       'Normal Fault', 'Active Fault','Magnetic Anomaly (nT)',
       'Vertical Gravity Gradient (E)', 'Free Air Anomaly (mGal)',
       'Bouguer Anomaly (mGal)', 'ID', 'Apparent Geothermal Gradient (°C/Km)']]

if TARGET_COL not in df_final.columns:
    candidates = [c for c in df_final.columns if 'grad' in c.lower() or 'geother' in c.lower()]
    if candidates:
        TARGET_COL = candidates[0]
        print("TARGET_COL no encontrado; usando:", TARGET_COL)
    else:
        raise ValueError("TARGET_COL no encontrado. Ajusta TARGET_COL en Cell 2.")

# Selección inicial de columnas tabulares (excluye ID y target)
tabular_cols = [c for c in df_final.columns if c not in ['ID', TARGET_COL]]

# Forzar a numérico
for c in tabular_cols:
    df_final[c] = pd.to_numeric(df_final[c], errors='coerce')

# Excluir explícitamente coordenadas (lat/lon)
geo_keywords = ['lati', 'longi', 'long', 'latitude', 'longitude']
geo_cols = [c for c in tabular_cols if any(k in c.lower() for k in geo_keywords)]
if len(geo_cols) > 0:
    print("Columnas geográficas detectadas y excluidas de features tabulares:", geo_cols)
tabular_cols = [c for c in tabular_cols if c not in geo_cols]

# Mantener solo columnas numéricas no vacías
tabular_cols = [c for c in tabular_cols if pd.api.types.is_numeric_dtype(df_final[c]) and not df_final[c].isna().all()]

print("Tabular columns used (sin lat/lon):", tabular_cols)

# Split train / test (80/20)
train_df, test_df, weights_train, _= train_test_split(df_final, weights, test_size=0.2, random_state=42)
# train_df = train_df.reset_index(drop=True)
# test_df  = test_df.reset_index(drop=True)

print("train/test sizes:", len(train_df), len(test_df), len(weights_train))

# Imputación tabular
if TAB_IMPUTE_METHOD == 'median':
    med = train_df[tabular_cols].median()
    train_df[tabular_cols] = train_df[tabular_cols].fillna(med)
    test_df[tabular_cols]  = test_df[tabular_cols].fillna(med)
elif TAB_IMPUTE_METHOD == 'knn':
    knn_tab = KNNImputer(n_neighbors=K_NEIGHBORS)
    train_df[tabular_cols] = knn_tab.fit_transform(train_df[tabular_cols])
    test_df[tabular_cols]  = knn_tab.transform(test_df[tabular_cols])
else:
    raise ValueError("TAB_IMPUTE_METHOD debe ser 'median' o 'knn'")

train_df[TARGET_COL + "_raw"] = train_df[TARGET_COL].copy()
test_df[TARGET_COL + "_raw"]  = test_df[TARGET_COL].copy()

# Manejo de posibles valores negativos que impidan log1p
min_train_target = train_df[TARGET_COL].min()
target_shift = 0.0
if np.isnan(min_train_target):
    raise ValueError("TARGET contiene solo NaNs en train (revisa imputación).")
if min_train_target <= -1.0:
    # aplicamos shift para que todos los valores sean >= -1 antes de log1p
    target_shift = float(abs(min_train_target) + 1.0)
    print(f"Aplicando target_shift = {target_shift:.6f} antes de log1p (min_train_target={min_train_target:.6f})")
    train_df[TARGET_COL] = train_df[TARGET_COL] + target_shift
    test_df[TARGET_COL]  = test_df[TARGET_COL] + target_shift

# Aplicar log1p
train_df[TARGET_COL] = np.log1p(train_df[TARGET_COL].astype(float))
test_df[TARGET_COL]  = np.log1p(test_df[TARGET_COL].astype(float))

# StandarScaler para el target (fit SOLO en train)
target_scaler = StandardScaler()
train_df[[TARGET_COL]] = target_scaler.fit_transform(train_df[[TARGET_COL]])
test_df[[TARGET_COL]]  = target_scaler.transform(test_df[[TARGET_COL]])

# Guardar scaler y shift para invertir luego
joblib.dump(target_scaler, OUT_DIR / "target_scaler.pkl")
joblib.dump({"shift": float(target_shift)}, OUT_DIR / "target_shift.json")

# Estandarización
if len(tabular_cols) > 0:
    tab_scaler = StandardScaler()
    train_df[tabular_cols] = tab_scaler.fit_transform(train_df[tabular_cols])
    test_df[tabular_cols]  = tab_scaler.transform(test_df[tabular_cols])
    joblib.dump(tab_scaler, OUT_DIR / "tabular_scaler.pkl")
else:
    print("No hay columnas tabulares para estandarizar (tabular_cols está vacío).")

Columnas geográficas detectadas y excluidas de features tabulares: ['Latitude', 'Longitude']
Tabular columns used (sin lat/lon): ['Elevation (m)', 'Curie Depth (Km)', 'Moho Depth (m)', 'Strike-slip Fault', 'Reverse or Thrust Fault', 'Right-lateral Fault', 'Nearest Basement', 'Normal Fault', 'Active Fault', 'Magnetic Anomaly (nT)', 'Vertical Gravity Gradient (E)', 'Free Air Anomaly (mGal)', 'Bouguer Anomaly (mGal)']
train/test sizes: 3560 891 3560


In [ ]:
# Cell 6
class MultimodalDataset(Dataset):
    def __init__(self, df, base_dir, tabular_cols, target_col,
                 weights=None,
                 fill_method=FILL_METHOD_IMAGE, k=K_NEIGHBORS, image_norm=IMAGE_NORM, target_size=TARGET_SIZE):
        self.df = df.reset_index(drop=True)
        self.base_dir = Path(base_dir)
        self.tabular_cols = tabular_cols
        self.target_col = target_col
        self.fill_method = fill_method
        self.k = k
        self.image_norm = image_norm
        self.target_size = target_size

        # weights: array-like alineado con df (opcional)
        if weights is not None:
            w = np.asarray(weights)
            if len(w) != len(self.df):
                raise ValueError("weights length must match dataframe length")
            self.weights = w.astype(np.float32)
        else:
            self.weights = None

    def __len__(self):
        return len(self.df)

    def _read_preprocess(self, idv):
        iv = int(float(idv))
        tif_path = self.base_dir / f"LANDSAT7_{iv}.tif"
        if not tif_path.exists():
            raise FileNotFoundError(f"No existe {tif_path}")
        arr, _ = read_geotiff(tif_path)
        if self.fill_method == 'median':
            arr = fill_missing_median(arr)
        elif self.fill_method == 'knn':
            arr = fill_missing_knn_mean(arr, k=self.k)
        arr = resize_multiband(arr, target_size=self.target_size)
        if self.image_norm == 'minmax':
            arr = normalize_image_minmax(arr)
        elif self.image_norm == 'zscore':
            arr = normalize_image_zscore(arr)
        img_tensor = torch.from_numpy(arr).float()
        return img_tensor

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = self._read_preprocess(row['ID'])

        tabs = row[self.tabular_cols].astype(np.float32).values
        tab_tensor = torch.from_numpy(tabs)

        target = np.float32(row[self.target_col])
        sample = {
            "image": img,
            "tabular": tab_tensor,
            "target": torch.tensor(target, dtype=torch.float32),
            "id": int(row['ID']),
        }

        return sample

In [ ]:
# Cell 7
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
    def forward(self, x):
        # x: (B, C)
        z = F.relu(self.fc1(x))
        w = torch.sigmoid(self.fc2(z))  # (B, C)
        return x * w

def get_resnet50_backbone(in_channels, pretrained=True):
    try:
        if pretrained and hasattr(models, "ResNet50_Weights"):
            res = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        else:
            res = models.resnet50(pretrained=pretrained)
    except Exception:
        res = models.resnet50(pretrained=pretrained)

    if in_channels != 3:
        old = res.conv1
        new_conv = nn.Conv2d(in_channels, old.out_channels, kernel_size=old.kernel_size,
                             stride=old.stride, padding=old.padding, bias=old.bias is not None)
        with torch.no_grad():
            w = old.weight.clone()
            w_mean = w.mean(dim=1, keepdim=True)
            new_conv.weight.copy_(w_mean.repeat(1, in_channels, 1, 1))
            if old.bias is not None:
                new_conv.bias.copy_(old.bias)
        res.conv1 = new_conv

    backbone = nn.Sequential(*list(res.children())[:-1])
    feat_dim = res.fc.in_features
    return backbone, feat_dim

class Resnet50_SE_Multimodal(nn.Module):
    def __init__(self, in_channels, n_tabular, pretrained=True, freeze_backbone=False, se_reduction=16):
        super().__init__()
        self.backbone, feat_dim = get_resnet50_backbone(in_channels, pretrained=pretrained)
        self.se = SEBlock(feat_dim, reduction=se_reduction)
        self.img_fc = nn.Linear(feat_dim, 256)
        self.tab_fc = nn.Sequential(nn.Linear(n_tabular, 128), nn.ReLU(), nn.Linear(128,128), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(256+128,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,1))
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, img, tab):
        x = self.backbone(img)
        x = x.view(x.size(0), -1)
        x = self.se(x)
        x = self.img_fc(x)
        t = self.tab_fc(tab)
        z = torch.cat([x,t], dim=1)
        out = self.head(z)
        return out.view(-1)

In [ ]:
# Cell 8

from torch.utils.data import WeightedRandomSampler

sampler = WeightedRandomSampler(
    weights=weights_train,
    num_samples=int(1.5*len(weights_train)),
    replacement=True
)

weights_train = np.asarray(weights_train).astype(np.float32)

train_ds = MultimodalDataset(train_df, BASE_DIR, tabular_cols, TARGET_COL, weights=None,
                             fill_method=FILL_METHOD_IMAGE, k=K_NEIGHBORS, image_norm=IMAGE_NORM, target_size=TARGET_SIZE)
test_ds  = MultimodalDataset(test_df,  BASE_DIR, tabular_cols, TARGET_COL, weights=None,
                             fill_method=FILL_METHOD_IMAGE, k=K_NEIGHBORS, image_norm=IMAGE_NORM, target_size=TARGET_SIZE)

train_loader = DataLoader(train_ds, sampler=sampler, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

def weighted_mse_loss(preds, targets, weights, eps=1e-8):
    if weights is None:
        return torch.mean((preds - targets)**2)
    w = weights.view_as(preds).float()
    num = torch.sum(w * (preds - targets)**2)
    denom = torch.sum(w) + eps
    return num / denom

n_tabular = len(tabular_cols)
n_bands = 7

model = Resnet50_SE_Multimodal(in_channels=n_bands, n_tabular=n_tabular, pretrained=True, freeze_backbone=True).to(DEVICE)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = torch.nn.SmoothL1Loss()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 104MB/s]


In [ ]:
# Cell 9: entrenar cabeza
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

best_test_rmse = np.inf

for ep in range(1, EPOCHS_HEAD+1):
    # Entrenamiento
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Head Epoch {ep}", leave=False)
    for batch in pbar:
        imgs = batch['image'].to(DEVICE)
        tabs = batch['tabular'].to(DEVICE).float()
        targets = batch['target'].to(DEVICE).float()

        optimizer.zero_grad()
        batch_weights = batch.get('weight', None)
        if batch_weights is not None:
            batch_weights = batch_weights.to(DEVICE).float()
        preds = model(imgs, tabs)
        loss = weighted_mse_loss(preds, targets, batch_weights)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        pbar.set_postfix({"loss": np.mean(train_losses)})
    scheduler.step()

    # Métricas
    model.eval()
    train_preds = []
    train_targs = []
    with torch.no_grad():
        for batch in train_loader:
            imgs = batch['image'].to(DEVICE)
            tabs = batch['tabular'].to(DEVICE).float()
            targets = batch['target'].to(DEVICE).float()
            preds = model(imgs, tabs)
            train_preds.append(preds.cpu().numpy())
            train_targs.append(targets.cpu().numpy())

    if len(train_preds) > 0:
        train_preds_arr = np.concatenate(train_preds)
        train_targs_arr = np.concatenate(train_targs)
        tr_rmse = np.sqrt(np.mean((train_targs_arr - train_preds_arr)**2))
        tr_mae  = mean_absolute_error(train_targs_arr, train_preds_arr)
        try:
            tr_r2 = r2_score(train_targs_arr, train_preds_arr)
        except Exception:
            tr_r2 = float('nan')
    else:
        tr_rmse = tr_mae = tr_r2 = float('nan')

    print(f"[Head] Epoch {ep}/{EPOCHS_HEAD}  TrainLoss={np.mean(train_losses):.4f}  TrainRMSE={tr_rmse:.4f}  TrainMAE={tr_mae:.4f}  TrainR2={tr_r2:.4f}")

    #Métricas sobre test
    preds_list, targs_list = [], []
    ids_list = []
    with torch.no_grad():
        for batch in test_loader:
            imgs = batch['image'].to(DEVICE)
            tabs = batch['tabular'].to(DEVICE).float()
            targets = batch['target'].to(DEVICE).float()
            preds = model(imgs, tabs)
            preds_list.append(preds.cpu().numpy())
            targs_list.append(targets.cpu().numpy())
            ids_list.extend(batch['id'])

    if len(preds_list) > 0:
        preds_arr = np.concatenate(preds_list)
        targs_arr = np.concatenate(targs_list)
        test_rmse = np.sqrt(np.mean((targs_arr - preds_arr)**2))
        test_mae  = mean_absolute_error(targs_arr, preds_arr)
        try:
            test_r2 = r2_score(targs_arr, preds_arr)
        except Exception:
            test_r2 = float('nan')
    else:
        test_rmse = test_mae = test_r2 = float('nan')

    print(f"       TestRMSE={test_rmse:.4f}  TestMAE={test_mae:.4f}  TestR2={test_r2:.4f}")

    # Guardar mejor según test
    if test_rmse < best_test_rmse - 1e-6:
        best_test_rmse = test_rmse
        torch.save({"epoch": ep, "model": model.state_dict(), "opt": optimizer.state_dict()}, OUT_DIR / "best_head.pth")
        print("  -> Mejor modelo (head) guardado.")


[Head] Epoch 1/3  TrainLoss=0.6781  TrainRMSE=0.8074  TrainMAE=0.5891  TrainR2=0.4192
       TestRMSE=0.7546  TestMAE=0.5460  TestR2=0.4378
  -> Mejor modelo (head) guardado.


[Head] Epoch 2/3  TrainLoss=0.5984  TrainRMSE=0.7786  TrainMAE=0.5623  TrainR2=0.4772
       TestRMSE=0.7289  TestMAE=0.5288  TestR2=0.4755
  -> Mejor modelo (head) guardado.


[Head] Epoch 3/3  TrainLoss=0.5690  TrainRMSE=0.7599  TrainMAE=0.5481  TrainR2=0.5063
       TestRMSE=0.7279  TestMAE=0.5196  TestR2=0.4769
  -> Mejor modelo (head) guardado.


In [ ]:
# Cell 10: unfreeze backbone
ck = torch.load(OUT_DIR / "best_head.pth", map_location=DEVICE)
model.load_state_dict(ck["model"])
print("Cargado checkpoint head")

# Unfreeze backbone params
for p in model.backbone.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_test_rmse_backbone = best_test_rmse  # Viene de la fase head

for ep in range(1, EPOCHS_BACKBONE+1):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Backbone Epoch {ep}", leave=False)
    for batch in pbar:
        imgs = batch['image'].to(DEVICE)
        tabs = batch['tabular'].to(DEVICE).float()
        targets = batch['target'].to(DEVICE).float()

        optimizer.zero_grad()
        batch_weights = batch.get('weight', None)
        if batch_weights is not None:
            batch_weights = batch_weights.to(DEVICE).float()
        preds = model(imgs, tabs)
        loss = weighted_mse_loss(preds, targets, batch_weights)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        pbar.set_postfix({"loss": np.mean(train_losses)})
    scheduler.step()

    # Métricas
    model.eval()
    train_preds = []
    train_targs = []
    with torch.no_grad():
        for batch in train_loader:
            imgs = batch['image'].to(DEVICE)
            tabs = batch['tabular'].to(DEVICE).float()
            targets = batch['target'].to(DEVICE).float()
            preds = model(imgs, tabs)
            train_preds.append(preds.cpu().numpy())
            train_targs.append(targets.cpu().numpy())

    if len(train_preds) > 0:
        train_preds_arr = np.concatenate(train_preds)
        train_targs_arr = np.concatenate(train_targs)
        tr_rmse = np.sqrt(np.mean((train_targs_arr - train_preds_arr)**2))
        tr_mae  = mean_absolute_error(train_targs_arr, train_preds_arr)
        try:
            tr_r2 = r2_score(train_targs_arr, train_preds_arr)
        except Exception:
            tr_r2 = float('nan')
    else:
        tr_rmse = tr_mae = tr_r2 = float('nan')

    print(f"[Backbone] Epoch {ep}/{EPOCHS_BACKBONE}  TrainLoss={np.mean(train_losses):.4f}  "
          f"TrainRMSE={tr_rmse:.4f}  TrainMAE={tr_mae:.4f}  TrainR2={tr_r2:.4f}")

    # Métricas sobre test
    preds_list, targs_list = [], []
    ids_list = []
    with torch.no_grad():
        for batch in test_loader:
            imgs = batch['image'].to(DEVICE)
            tabs = batch['tabular'].to(DEVICE).float()
            targets = batch['target'].to(DEVICE).float()
            preds = model(imgs, tabs)
            preds_list.append(preds.cpu().numpy())
            targs_list.append(targets.cpu().numpy())
            ids_list.extend(batch['id'])

    if len(preds_list) > 0:
        preds_arr = np.concatenate(preds_list)
        targs_arr = np.concatenate(targs_list)
        test_rmse = np.sqrt(np.mean((targs_arr - preds_arr)**2))
        test_mae  = mean_absolute_error(targs_arr, preds_arr)
        try:
            test_r2 = r2_score(targs_arr, preds_arr)
        except Exception:
            test_r2 = float('nan')
    else:
        test_rmse = test_mae = test_r2 = float('nan')

    print(f"         TestRMSE={test_rmse:.4f}  TestMAE={test_mae:.4f}  TestR2={test_r2:.4f}")

    # Guardar mejor según test
    if test_rmse < best_test_rmse_backbone - 1e-6:
        best_test_rmse_backbone = test_rmse
        torch.save({"epoch": ep, "model": model.state_dict(), "opt": optimizer.state_dict()}, OUT_DIR / "best_backbone.pth")
        print("  -> Mejor modelo (backbone) guardado.")


Cargado checkpoint head


[Backbone] Epoch 1/4  TrainLoss=0.5033  TrainRMSE=0.7052  TrainMAE=0.5168  TrainR2=0.5636
         TestRMSE=0.7324  TestMAE=0.5264  TestR2=0.4705


[Backbone] Epoch 2/4  TrainLoss=0.4336  TrainRMSE=0.6261  TrainMAE=0.4514  TrainR2=0.6458
         TestRMSE=0.7287  TestMAE=0.5260  TestR2=0.4758


[Backbone] Epoch 3/4  TrainLoss=0.3439  TrainRMSE=0.8171  TrainMAE=0.4617  TrainR2=0.4035
         TestRMSE=0.7691  TestMAE=0.5570  TestR2=0.4160


[Backbone] Epoch 4/4  TrainLoss=0.2815  TrainRMSE=0.5615  TrainMAE=0.4068  TrainR2=0.7198
         TestRMSE=0.7449  TestMAE=0.5390  TestR2=0.4522


In [ ]:
# Cell 11: full

# Cargar mejor checkpoint previo (backbone o head)
ck_path = None
for p in ["best_backbone.pth", "best_head.pth"]:
    if (OUT_DIR / p).exists():
        ck_path = OUT_DIR / p
        break
if ck_path is None:
    raise FileNotFoundError("No se encontró checkpoint para arrancar full fine-tune final.")
ck = torch.load(OUT_DIR / ck_path.name, map_location=DEVICE)
model.load_state_dict(ck["model"])
print("Loaded checkpoint for final fine-tune:", ck_path)

# Desbloquear todos los parámetros para fine-tune completo
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=LR_FINE/5, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_test_rmse_full = np.inf

for ep in range(1, EPOCHS_FULL+1):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Full Epoch {ep}", leave=False)
    for batch in pbar:
        imgs = batch['image'].to(DEVICE)
        tabs = batch['tabular'].to(DEVICE).float()
        targets = batch['target'].to(DEVICE).float()

        optimizer.zero_grad()
        batch_weights = batch.get('weight', None)
        if batch_weights is not None:
            batch_weights = batch_weights.to(DEVICE).float()
        preds = model(imgs, tabs)
        loss = weighted_mse_loss(preds, targets, batch_weights)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        pbar.set_postfix({"loss": np.mean(train_losses)})
    scheduler.step()

    model.eval()
    train_preds = []
    train_targs = []
    with torch.no_grad():
        for batch in train_loader:
            imgs = batch['image'].to(DEVICE)
            tabs = batch['tabular'].to(DEVICE).float()
            targets = batch['target'].to(DEVICE).float()
            preds = model(imgs, tabs)
            train_preds.append(preds.cpu().numpy())
            train_targs.append(targets.cpu().numpy())

    if len(train_preds) > 0:
        train_preds_arr = np.concatenate(train_preds)
        train_targs_arr = np.concatenate(train_targs)
        tr_rmse = np.sqrt(np.mean((train_targs_arr - train_preds_arr)**2))
        tr_mae  = mean_absolute_error(train_targs_arr, train_preds_arr)
        try:
            tr_r2 = r2_score(train_targs_arr, train_preds_arr)
        except Exception:
            tr_r2 = float('nan')
    else:
        tr_rmse = tr_mae = tr_r2 = float('nan')

    print(f"[Full] Epoch {ep}/{EPOCHS_FULL}  TrainLoss={np.mean(train_losses):.4f}  "
          f"TrainRMSE={tr_rmse:.4f}  TrainMAE={tr_mae:.4f}  TrainR2={tr_r2:.4f}")

    preds_list, targs_list = [], []
    ids_list = []
    with torch.no_grad():
        for batch in test_loader:
            imgs = batch['image'].to(DEVICE)
            tabs = batch['tabular'].to(DEVICE).float()
            targets = batch['target'].to(DEVICE).float()
            preds = model(imgs, tabs)
            preds_list.append(preds.cpu().numpy())
            targs_list.append(targets.cpu().numpy())
            ids_list.extend(batch['id'])

    if len(preds_list) > 0:
        preds_arr = np.concatenate(preds_list)
        targs_arr = np.concatenate(targs_list)
        test_rmse = np.sqrt(np.mean((targs_arr - preds_arr)**2))
        test_mae  = mean_absolute_error(targs_arr, preds_arr)
        try:
            test_r2 = r2_score(targs_arr, preds_arr)
        except Exception:
            test_r2 = float('nan')
    else:
        test_rmse = test_mae = test_r2 = float('nan')

    print(f"         TestRMSE={test_rmse:.4f}  TestMAE={test_mae:.4f}  TestR2={test_r2:.4f}")

    if test_rmse < best_test_rmse_full - 1e-6:
        best_test_rmse_full = test_rmse
        torch.save({"epoch": ep, "model": model.state_dict(), "opt": optimizer.state_dict()}, OUT_DIR / "best_full.pth")
        print("  -> Mejor modelo (full) guardado.")


Loaded checkpoint for final fine-tune: /content/run_resnet50_se_maps/best_head.pth


[Full] Epoch 1/3  TrainLoss=0.5363  TrainRMSE=0.8812  TrainMAE=0.5377  TrainR2=0.3308
         TestRMSE=0.7352  TestMAE=0.5292  TestR2=0.4664
  -> Mejor modelo (full) guardado.


[Full] Epoch 2/3  TrainLoss=0.5023  TrainRMSE=0.6863  TrainMAE=0.4984  TrainR2=0.5942
         TestRMSE=0.7159  TestMAE=0.5136  TestR2=0.4940
  -> Mejor modelo (full) guardado.


[Full] Epoch 3/3  TrainLoss=0.4533  TrainRMSE=0.6738  TrainMAE=0.4671  TrainR2=0.5933
         TestRMSE=0.7167  TestMAE=0.5175  TestR2=0.4929


In [ ]:
# Cell 12: métricas normalizadas y desnormalizadas

# Cargar scaler y shift del target
target_scaler = joblib.load(OUT_DIR / "target_scaler.pkl")
target_shift = joblib.load(OUT_DIR / "target_shift.json")["shift"]


# Función para invertir transformaciones
def inverse_target_transform(y_scaled, scaler, shift):
    y_scaled = np.array(y_scaled).reshape(-1, 1)

    # Inversa RobustScaler
    y_log = scaler.inverse_transform(y_scaled).ravel()

    # Inversa log1p
    y = np.expm1(y_log)

    # Quitar shift si existía
    if shift != 0:
        y = y - shift

    return y


#  Cargar mejor checkpoint
best_path = None
for p in ["best_full.pth", "best_backbone.pth", "best_head.pth"]:
    candidate = OUT_DIR / p
    if candidate.exists():
        best_path = candidate
        break

if best_path is None:
    raise FileNotFoundError("No se encontró checkpoint guardado.")

ck = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ck["model"])
print("Checkpoint cargado:", best_path, "epoch:", ck.get("epoch", "unk"))

model.eval()


# Evaluación en entrenamiento
train_preds, train_targs, ids_train= [], [], []

with torch.no_grad():
    for batch in tqdm(train_loader, desc="Train"):
        imgs = batch['image'].to(DEVICE)
        tabs = batch['tabular'].to(DEVICE).float()

        out = model(imgs, tabs).cpu().numpy()

        train_preds.append(out)
        train_targs.append(batch['target'].cpu().numpy())
        ids_train.extend(batch['id'])

train_preds = np.concatenate(train_preds)
train_targs = np.concatenate(train_targs)

# Métricas en escala transformada
train_rmse_scaled = np.sqrt(np.mean((train_preds-train_targs)**2))
train_mae_scaled  = np.mean(np.abs(train_preds-train_targs))

# Invertir escala
train_preds_real = inverse_target_transform(train_preds, target_scaler, target_shift)
train_targs_real = inverse_target_transform(train_targs, target_scaler, target_shift)

train_rmse = np.sqrt(np.mean((train_preds_real-train_targs_real)**2))
train_mae  = np.mean(np.abs(train_preds_real-train_targs_real))

ss_res = np.sum((train_targs_real-train_preds_real)**2)
ss_tot = np.sum((train_targs_real-np.mean(train_targs_real))**2)
train_r2 = 1-ss_res/ss_tot if ss_tot!=0 else 0

print("\n===== TRAIN =====")
print("Escala transformada:")
print("RMSE: %.4f  MAE: %.4f" % (train_rmse_scaled, train_mae_scaled))

print("\nEscala REAL:")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (train_rmse, train_mae, train_r2))


# Evaluación en test
test_preds, test_targs, ids = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test"):
        imgs = batch['image'].to(DEVICE)
        tabs = batch['tabular'].to(DEVICE).float()

        out = model(imgs, tabs).cpu().numpy()

        test_preds.append(out)
        test_targs.append(batch['target'].cpu().numpy())
        ids.extend(batch['id'])

test_preds = np.concatenate(test_preds)
test_targs = np.concatenate(test_targs)

# Métricas transformadas
test_rmse_scaled = np.sqrt(np.mean((test_preds-test_targs)**2))
test_mae_scaled  = np.mean(np.abs(test_preds-test_targs))

# Escala real
test_preds_real = inverse_target_transform(test_preds, target_scaler, target_shift)
test_targs_real = inverse_target_transform(test_targs, target_scaler, target_shift)

test_rmse = np.sqrt(np.mean((test_preds_real-test_targs_real)**2))
test_mae  = np.mean(np.abs(test_preds_real-test_targs_real))

ss_res = np.sum((test_targs_real-test_preds_real)**2)
ss_tot = np.sum((test_targs_real-np.mean(test_targs_real))**2)
test_r2 = 1-ss_res/ss_tot if ss_tot!=0 else 0

print("\n===== TEST =====")
print("Escala transformada:")
print("RMSE: %.4f  MAE: %.4f" % (test_rmse_scaled, test_mae_scaled))

print("\nEscala REAL:")
print("RMSE: %.4f  MAE: %.4f  R2: %.4f" % (test_rmse, test_mae, test_r2))


Checkpoint cargado: /content/run_resnet50_se_maps/best_full.pth epoch: 2


Train: 100%|██████████| 668/668 [07:46<00:00,  1.43it/s]



===== TRAIN =====
Escala transformada:
RMSE: 0.6927  MAE: 0.4991

Escala REAL:
RMSE: 3.8066  MAE: 2.6004  R2: 0.5650


Test: 100%|██████████| 112/112 [01:12<00:00,  1.54it/s]


===== TEST =====
Escala transformada:
RMSE: 0.7159  MAE: 0.5136

Escala REAL:
RMSE: 3.8370  MAE: 2.5901  R2: 0.4567


In [ ]:
# !zip -j best_full.zip /content/run_resnet50_se_maps/best_full.pth
# from google.colab import files
# files.download("best_full.zip")


In [ ]:
# !zip -j best_head.zip /content/run_resnet50_se_maps/best_head.pth
# from google.colab import files
# files.download("best_head.zip")


In [ ]:
# !zip -j best_backbone.zip /content/run_resnet50_se_maps/best_backbone.pth
# from google.colab import files
# files.download("best_backbone.zip")

In [ ]:
# Cell 13: Mapas espaciales para el conjunto de test

# DataFrame con valores reales (desnormalizados)
df_pred = pd.DataFrame({
    "ID": ids,
    "actual_real": test_targs_real,
    "pred_real": test_preds_real
})

# GeoDataFrame actual
actual_data = test_df.copy()
actual_data['ID'] = actual_data['ID'].astype(int)
df_pred['ID'] = df_pred['ID'].astype(int)

actual_gdf = gpd.GeoDataFrame(
    actual_data.merge(df_pred[['ID','actual_real','pred_real']], on='ID'),
    geometry=gpd.points_from_xy(
        actual_data['Longitude'],
        actual_data['Latitude']
    ),
    crs="EPSG:4326"
).dropna(subset=['geometry'])

# Predicho (mismo dataset)
predicted_gdf = actual_gdf.copy()

# Diferencias en unidades reales
actual_col = "actual_real"
pred_col   = "pred_real"

actual_gdf["diff"] = actual_gdf[actual_col] - actual_gdf[pred_col]
actual_gdf["abs_diff"] = np.abs(actual_gdf["diff"])

merged = actual_gdf  # simplificación

# Rangos de colores coherentes
vmin = min(merged[actual_col].min(), merged[pred_col].min())
vmax = max(merged[actual_col].max(), merged[pred_col].max())

vabs = np.nanmax(np.abs(merged["diff"]))
vabs_abs = merged["abs_diff"].max()

# Mapa base Colombia
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

possible_cols = ['iso_a3', 'ISO_A3', 'ADM0_A3', 'SOV_A3']
code_col = next(c for c in possible_cols if c in world.columns)

colombia = world[world[code_col] == 'COL']

minx, miny, maxx, maxy = colombia.total_bounds
padx = (maxx - minx) * 0.05
pady = (maxy - miny) * 0.05
bbox = (minx - padx, maxx + padx, miny - pady, maxy + pady)

def base_map(ax):
    colombia.boundary.plot(ax=ax, linewidth=1.0, color='black')
    ax.set_xlim(bbox[0], bbox[1])
    ax.set_ylim(bbox[2], bbox[3])
    ax.set_aspect('auto')
    ax.grid(True, linestyle='--', linewidth=0.3)

# 1 - Mapa real
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column=actual_col, cmap='jet',
    markersize=60, legend=True, vmin=vmin, vmax=vmax
)
fig.savefig("map_actual_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# 2 - Mapa predicho
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column=pred_col, cmap='jet',
    markersize=60, legend=True, vmin=vmin, vmax=vmax
)
fig.savefig("map_predicted_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# 3 - Mapa diferencia
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="diff", cmap='RdBu_r',
    markersize=60, legend=True, vmin=-vabs, vmax=vabs
)
fig.savefig("map_difference_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# 4 - Mapa diferencia absoluta
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="abs_diff", cmap='viridis',
    markersize=60, legend=True, vmin=0, vmax=vabs_abs
)
fig.savefig("map_abs_difference_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 14: Mapas espaciales para el conjunto de entrenamiento

# DataFrame con valores reales desnormalizados
df_pred_train = pd.DataFrame({
    "ID": ids_train,              # ← IDs del train_loader
    "actual_real": train_targs_real,
    "pred_real": train_preds_real
})

df_pred_train['ID'] = df_pred_train['ID'].astype(int)

# GeoDataFrame base con coordenadas de entrenamiento
actual_data = train_df.copy()
actual_data['ID'] = actual_data['ID'].astype(int)

merged = actual_data.merge(df_pred_train, on='ID')

merged = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(
        merged['Longitude'],
        merged['Latitude']
    ),
    crs="EPSG:4326"
).dropna(subset=['geometry'])

# Diferencias reales
merged["diff"] = merged["actual_real"] - merged["pred_real"]
merged["abs_diff"] = np.abs(merged["diff"])

# Rangos color consistentes
vmin = min(merged["actual_real"].min(), merged["pred_real"].min())
vmax = max(merged["actual_real"].max(), merged["pred_real"].max())

vabs = np.nanmax(np.abs(merged["diff"]))
vabs_abs = merged["abs_diff"].max()

# Mapa de Colombia
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

possible_cols = ['iso_a3', 'ISO_A3', 'ADM0_A3', 'SOV_A3']
code_col = next(c for c in possible_cols if c in world.columns)

colombia = world[world[code_col] == 'COL']

minx, miny, maxx, maxy = colombia.total_bounds
padx = (maxx - minx) * 0.05
pady = (maxy - miny) * 0.05
bbox = (minx - padx, maxx + padx, miny - pady, maxy + pady)

def base_map(ax):
    colombia.boundary.plot(ax=ax, linewidth=1.0, color='black')
    ax.set_xlim(bbox[0], bbox[1])
    ax.set_ylim(bbox[2], bbox[3])
    ax.set_aspect('auto')
    ax.grid(True, linestyle='--', linewidth=0.3)

# 1 - Actual
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="actual_real", cmap='jet',
    markersize=60, legend=True, vmin=vmin, vmax=vmax
)
fig.savefig("map_actual_train_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# 2 - Predicho
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="pred_real", cmap='jet',
    markersize=60, legend=True, vmin=vmin, vmax=vmax
)
fig.savefig("map_predicted_train_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# 3 - Diferencia
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="diff", cmap='RdBu_r',
    markersize=60, legend=True, vmin=-vabs, vmax=vabs
)
fig.savefig("map_difference_train_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# 4 - Diferencia absoluta
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="abs_diff", cmap='viridis',
    markersize=60, legend=True, vmin=0, vmax=vabs_abs
)
fig.savefig("map_abs_difference_train_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Cell 14: Mapas espaciales FULL dataset (TRAIN + TEST EN ESCALA REAL)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

# =====================================================
# 1. Combinar datasets originales
# =====================================================
full_df = pd.concat([train_df, test_df], ignore_index=True)
full_df["ID"] = full_df["ID"].astype(int)

# =====================================================
# 2. Combinar predicciones reales
# =====================================================
df_pred_all = pd.concat(
    [
        df_pred_train[["ID", "actual_real", "pred_real"]],
        df_pred[["ID", "actual_real", "pred_real"]]
    ],
    ignore_index=True
)

df_pred_all["ID"] = df_pred_all["ID"].astype(int)

# =====================================================
# 3. Merge coords + valores reales
# =====================================================
merged = full_df.merge(df_pred_all, on="ID")

merged = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(
        merged['Longitude'],
        merged['Latitude']
    ),
    crs="EPSG:4326"
).dropna(subset=['geometry'])

# =====================================================
# 4. Diferencias reales
# =====================================================
merged["diff"] = merged["actual_real"] - merged["pred_real"]
merged["abs_diff"] = np.abs(merged["diff"])

# =====================================================
# 5. Rangos color coherentes
# =====================================================
vmin = min(merged["actual_real"].min(), merged["pred_real"].min())
vmax = max(merged["actual_real"].max(), merged["pred_real"].max())

vabs = np.nanmax(np.abs(merged["diff"]))
vabs_abs = merged["abs_diff"].max()

# =====================================================
# 6. Base Colombia
# =====================================================
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

possible_cols = ['iso_a3', 'ISO_A3', 'ADM0_A3', 'SOV_A3']
code_col = next(c for c in possible_cols if c in world.columns)

colombia = world[world[code_col] == 'COL']

minx, miny, maxx, maxy = colombia.total_bounds
padx = (maxx - minx) * 0.05
pady = (maxy - miny) * 0.05
bbox = (minx - padx, maxx + padx, miny - pady, maxy + pady)

def base_map(ax):
    colombia.boundary.plot(ax=ax, linewidth=1.0, color='black')
    ax.set_xlim(bbox[0], bbox[1])
    ax.set_ylim(bbox[2], bbox[3])
    ax.set_aspect('auto')
    ax.grid(True, linestyle='--', linewidth=0.3)

# =====================================================
# 7️⃣ Actual REAL FULL
# =====================================================
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="actual_real", cmap='jet',
    markersize=60, legend=True, vmin=vmin, vmax=vmax
)
fig.savefig("map_actual_FULL_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# =====================================================
# 8️⃣ Predicho REAL FULL
# =====================================================
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="pred_real", cmap='jet',
    markersize=60, legend=True, vmin=vmin, vmax=vmax
)
fig.savefig("map_predicted_FULL_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# =====================================================
# 9️⃣ Diferencia REAL FULL
# =====================================================
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="diff", cmap='RdBu_r',
    markersize=60, legend=True, vmin=-vabs, vmax=vabs
)
fig.savefig("map_difference_FULL_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()

# =====================================================
# 🔟 Error absoluto REAL FULL
# =====================================================
fig, ax = plt.subplots(figsize=(10,8))
base_map(ax)
merged.plot(
    ax=ax, column="abs_diff", cmap='viridis',
    markersize=60, legend=True, vmin=0, vmax=vabs_abs
)
fig.savefig("map_abs_difference_FULL_REAL.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# Arrays EN ESCALA REAL
# =========================
tp = np.ravel(train_preds_real)
tt = np.ravel(train_targs_real)

ep = np.ravel(test_preds_real)
et = np.ravel(test_targs_real)

full_preds = np.concatenate([tp, ep])
full_targs = np.concatenate([tt, et])

datasets = {
    "TEST": (ep, et),
    "TRAIN": (tp, tt),
    "FULL": (full_preds, full_targs)
}

# =========================
# Rangos comunes ejes
# =========================
all_vals = np.concatenate([tp, tt, ep, et])
xmin = np.nanmin(all_vals)
xmax = np.nanmax(all_vals)

pad = (xmax - xmin) * 0.06 if (xmax - xmin) > 0 else 0.5
xlim = (xmin - pad, xmax + pad)
ylim = (xmin - pad, xmax + pad)

# =========================
# Colores elegantes estilo mapas
# =========================
point_color = "#1f4e5f"   # azul petróleo
line_color  = "#F54927"   # línea referencia
grid_color  = "#d9d9d9"

# =========================
# Graficar
# =========================
for name, (preds, targs) in datasets.items():

    preds = np.ravel(preds)
    targs = np.ravel(targs)

    rmse = np.sqrt(mean_squared_error(targs, preds))
    mae  = mean_absolute_error(targs, preds)
    r2   = r2_score(targs, preds) if len(targs) > 1 else np.nan

    fig, ax = plt.subplots(figsize=(8,8))

    # fondo paper
    ax.set_facecolor("#fafafa")

    ax.scatter(
        targs, preds,
        s=40,
        alpha=0.75,
        color=point_color,
        edgecolor="white",
        linewidth=0.4
    )

    # línea 1:1
    ax.plot(
        [xlim[0], xlim[1]],
        [xlim[0], xlim[1]],
        linestyle="--",
        color=line_color,
        linewidth=1.6
    )

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    ax.set_xlabel("Gradiente real (°C/km)", fontsize=12)
    ax.set_ylabel("Gradiente estimado (°C/km)", fontsize=12)

    ax.grid(True, linestyle="--", linewidth=0.6, color=grid_color)

    for spine in ax.spines.values():
        spine.set_color("#bbbbbb")
        spine.set_linewidth(0.8)

    fname = f"real_vs_estimado_{name.lower()}_REAL.pdf"
    fig.savefig(fname, dpi=300, bbox_inches='tight')

    print(
        f"{name} REAL -> RMSE={rmse:.3f}, "
        f"MAE={mae:.3f}, R2={r2:.3f}"
    )

    plt.show()
